# Weidner

## Index
1. [Instantiate model class](#Instantiate-model-class)
2. [Define clock metadata](#Define-clock-metadata)
3. [Download clock dependencies](#Download-clock-dependencies)
4. [Load features](#Load-features)
5. [Load weights into base model](#Load-weights-into-base-model)
6. [Load reference values](#Load-reference-values)
7. [Load preprocess and postprocess objects](#Load-preprocess-and-postprocess-objects)
8. [Check all clock parameters](#Check-all-clock-parameters)
9. [Normal feature ranges](#Normal-feature-ranges)
10. [Basic test](#Basic-test)
11. [Save torch model](#Save-torch-model)
12. [Clear directory](#Clear-directory)

Let's first import some packages:

In [1]:
import os
import inspect
import shutil
import json
import math
import torch
import pandas as pd
import pyaging as pya

## Instantiate model class

In [2]:
def print_entire_class(cls):
    source = inspect.getsource(cls)
    print(source)

print_entire_class(pya.models.Weidner)

class Weidner(pyagingModel):
    def __init__(self):
        super().__init__()

    def preprocess(self, x):
        if self.reference_values is None:
            return x
        if isinstance(self.reference_values, torch.Tensor):
            reference = self.reference_values.to(device=x.device, dtype=x.dtype)
        else:
            reference = torch.tensor(self.reference_values, device=x.device, dtype=x.dtype)
        return torch.where(torch.isnan(x), reference, x)

    def postprocess(self, x):
        return x



In [3]:
model = pya.models.Weidner()

## Define clock metadata

In [4]:
model.metadata["clock_name"] = "weidner"
model.metadata["data_type"] = "DNA methylation"  # Paper: DNA methylation changes at just three CpG sites
model.metadata["species"] = "Homo sapiens"  # Paper: blood samples from healthy human donors
model.metadata["year"] = 2014
model.metadata["approved_by_author"] = "⌛"
model.metadata["citation"] = "Weidner, C. I., Lin, Q., Koch, C. M., et al. (2014). Aging of blood can be tracked by DNA methylation changes at just three CpG sites. Genome Biology, 15, R24."
model.metadata["doi"] = "https://doi.org/10.1186/gb-2014-15-2-r24"
model.metadata["notes"] = "Three-site whole-blood epigenetic-age estimator. The sites were selected from Illumina 27K blood profiles, and the final multivariate linear equation was fitted on targeted bisulfite-pyrosequencing beta values from 82 blood samples and validated in 69 independent samples."
model.metadata["research_only"] = None
model.metadata["tissue"] = ["whole blood"]  # Paper: DNAm profiles derived from blood samples; model fitted on blood samples
model.metadata["predicts"] = ["biological age"]  # Paper: signature provides a simple biomarker to estimate the state of aging in blood
model.metadata["training_target"] = ["chronological age"]  # Paper: multivariate linear model to predict donor age
model.metadata["unit"] = ["years"]  # Paper: Predicted age (in years)
model.metadata["model_type"] = "linear regression"  # Paper: multivariate linear regression model
model.metadata["platform"] = ["Illumina 27K", "bisulfite sequencing"]  # Paper: HumanMethylation27 BeadChip feature selection followed by pyrosequencing after bisulfite conversion
model.metadata["population"] = "adults"  # Paper: 82 blood samples for training and independent validation set of 69 blood samples
model.metadata["journal"] = "Genome Biology"
model.metadata["last_author"] = "Wolfgang Wagner"
model.metadata["n_features"] = 3
model.metadata["citations"] = 973
model.metadata["citations_date"] = "2026-07-05"


## Download clock dependencies

In [5]:
supplementary_url = "https://raw.githubusercontent.com/bio-learn/biolearn/180852e2bab473303cb85da627178b1695ee9d86/biolearn/data/Weidner.csv"
supplementary_file_name = "Weidner.csv"
os.system(f"curl -sL -o {supplementary_file_name} {supplementary_url}")

0

## Load features

In [6]:
df = pd.read_csv('Weidner.csv')
model.features = df['CpGmarker'].tolist()

## Load weights into base model

In [7]:
weights = torch.tensor(df['CoefficientTraining'].tolist()).unsqueeze(0).float()
intercept = torch.tensor([38.0]).float()

In [8]:
base_model = pya.models.LinearModel(input_dim=len(model.features))

base_model.linear.weight.data = weights.float()
base_model.linear.bias.data = intercept.float()

model.base_model = base_model

## Load reference values

In [9]:
model.reference_values = None

## Load preprocess and postprocess objects

In [10]:
model.preprocess_name = None
model.preprocess_dependencies = None

In [11]:
model.postprocess_name = None
model.postprocess_dependencies = None

## Check all clock parameters

In [12]:
pya.utils.print_model_details(model)


%==================================== Model Details ====================================%
Model Attributes:

training: True
metadata: {'approved_by_author': '⌛',
 'citation': 'Weidner, Carola I., et al. "Aging of blood can be tracked by DNA '
             'methylation changes at just three CpG sites." Genome biology '
             '15.2 (2014): R24.',
 'clock_name': 'weidner',
 'data_type': 'methylation',
 'doi': 'https://doi.org/10.1186/gb-2014-15-2-r24',
 'notes': None,
 'research_only': None,
 'species': 'Homo sapiens',
 'version': None,
 'year': 2014}
reference_values: None
preprocess_name: None
preprocess_dependencies: None
postprocess_name: None
postprocess_dependencies: None
features: ['cg02228185', 'cg25809905', 'cg17861230']
base_model_features: None

%==================================== Model Details ====================================%
Model Structure:

base_model: LinearModel(
  (linear): Linear(in_features=3, out_features=1, bias=True)
)

%==============================

## Normal feature ranges

In [ ]:
# Units and plausibility ranges come from the package registry, keyed by feature name.
feature_ranges = pya.utils.resolve_feature_ranges(model.features, model.metadata["data_type"])
model.feature_units = [record["unit"] for record in feature_ranges]
pd.DataFrame.from_records(feature_ranges).head()

## Basic test

In [ ]:
# Exercise the clock with values in the middle of each feature's expected range.
records = pya.utils.resolve_feature_ranges(model.features, model.metadata["data_type"])
midpoints = [
    (record["low"] + record["high"]) / 2 if math.isfinite(record["high"]) else max(record["low"], 1.0)
    for record in records
]
input = torch.tensor([midpoints] * 10, dtype=torch.float64)
model.eval()
model.to(torch.float64)
pred = model(input)
pred

## Save torch model

In [14]:
torch.save(model, f"../weights/{model.metadata['clock_name']}.pt")

## Clear directory
<a id="10"></a>

In [15]:
# Function to remove a folder and all its contents
def remove_folder(path):
    try:
        shutil.rmtree(path)
        print(f"Deleted folder: {path}")
    except Exception as e:
        print(f"Error deleting folder {path}: {e}")

# Get a list of all files and folders in the current directory
all_items = os.listdir('.')

# Loop through the items
for item in all_items:
    # Check if it's a file and does not end with .ipynb
    if os.path.isfile(item) and not item.endswith('.ipynb'):
        os.remove(item)
        print(f"Deleted file: {item}")
    # Check if it's a folder
    elif os.path.isdir(item):
        remove_folder(item)

Deleted file: Weidner.csv
